A company allows only one entry per employee per day.
However, employees found a loophole:
they can enter multiple times using different email IDs.

You are given an entries table that logs:

employee name

address

email used

floor visited

resource used

🎯 Your task is to write an SQL query that returns:

For each person:

Total number of visits

Most visited floor

List of distinct resources used

🧱 Table Structure

CREATE TABLE entries ( 
    name VARCHAR(20),
    address VARCHAR(20),
    email VARCHAR(30),
    floor INT,
    resources VARCHAR(20)
);

In [0]:
%sql
WITH floor_visits AS (
    SELECT
        name,
        floor,
        COUNT(*) AS visit_count
    FROM entries
    GROUP BY name, floor
),

ranked_floor AS (
    SELECT
        name,
        floor,
        visit_count,
        ROW_NUMBER() OVER(
            PARTITION BY name
            ORDER BY visit_count DESC, floor
        ) AS rn
    FROM floor_visits
),

resource_list AS (
    SELECT
        name,
        GROUP_CONCAT(DISTINCT resources ORDER BY resources) AS resources_used
    FROM entries
    GROUP BY name
),

visit_count AS (
    SELECT
        name,
        COUNT(*) AS total_visits
    FROM entries
    GROUP BY name
)

SELECT
    v.name,
    v.total_visits,
    r.floor AS most_visited_floor,
    rl.resources_used
FROM visit_count v
JOIN ranked_floor r
    ON v.name = r.name
   AND r.rn = 1
JOIN resource_list rl
    ON v.name = rl.name
ORDER BY v.name;

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Total visits
visits = df.groupBy("name").agg(
    F.count("*").alias("total_visits")
)

# Resources used
resources = df.groupBy("name").agg(
    F.concat_ws(
        ",",
        F.sort_array(F.collect_set("resources"))
    ).alias("resources_used")
)

# Floor visit count
floor_visits = df.groupBy("name", "floor").agg(
    F.count("*").alias("visit_count")
)

# Rank floors
w = Window.partitionBy("name").orderBy(
    F.desc("visit_count"),
    F.asc("floor")
)

most_floor = (
    floor_visits
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select("name", F.col("floor").alias("most_visited_floor"))
)

# Final result
result = (
    visits
    .join(most_floor, "name")
    .join(resources, "name")
)

result.show(truncate=False)